In [ ]:
import cv2
import numpy as np
from PIL import Image
import supervision as sv
from rfdetr import RFDETRSegNano
from rfdetr.assets.coco_classes import COCO_CLASSES

# RF-DETR Segmentation 모델 로드
model = RFDETRSegNano()
model.optimize_for_inference()

cap = cv2.VideoCapture(0)

if cap.isOpened():
    print("실시간 세그멘테이션 시작. 종료하려면 'q'를 누르세요.")

    while True:
        ret, frame = cap.read()
        if not ret:
            print("프레임을 읽을 수 없습니다.")
            break

        # OpenCV frame은 BGR -> RGB 변환
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # RF-DETR 입력용 PIL 이미지 변환
        frame_pil = Image.fromarray(frame_rgb)

        # 추론
        dets = model.predict(frame_pil, threshold=0.5)

        # 라벨 생성
        labels = [
            f"{COCO_CLASSES[cid]} {conf:.2f}"
            for cid, conf in zip(dets.class_id, dets.confidence)
        ]

        # 어노테이션
        annotated = frame_rgb.copy()
        annotated = sv.MaskAnnotator().annotate(annotated, dets)
        annotated = sv.BoxAnnotator().annotate(annotated, dets)
        annotated = sv.LabelAnnotator().annotate(annotated, dets, labels)

        # OpenCV 출력용 RGB -> BGR 변환
        annotated_bgr = cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR)

        cv2.imshow("RF-DETR Segmentation Realtime", annotated_bgr)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

cap.release()
cv2.destroyAllWindows()